In [1]:
import pandas as pd
import numpy as np
import string
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import GradientBoostingClassifier  # Change this line to use a different ensemble model if needed
import nltk

# Download necessary NLTK data files
nltk.download('wordnet')

# Preprocessing function without removing stopwords
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# Load and prepare the training data
train_df = pd.read_csv("/Users/eren/Desktop/412Project/bugs-train.csv")
test_df = pd.read_csv("/Users/eren/Desktop/412Project/bugs-test.csv")
severity_mapping = {
    'enhancement': 1,
    'trivial': 2,
    'minor': 3,
    'normal': 4,
    'major': 5,
    'blocker': 6,
    'critical': 7
}
train_df['severity'] = train_df['severity'].map(severity_mapping)

# Apply preprocessing to the text data
train_df['summary'] = train_df['summary'].apply(preprocess_text)
test_df['summary'] = test_df['summary'].apply(preprocess_text)

# Prepare text data
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['summary'])
X_test_tfidf = vectorizer.transform(test_df['summary'])

# Train separate models for each class using stratified sampling
models = {}
for severity, label in severity_mapping.items():
    y_binary = train_df['severity'] == label
    X_train, X_val, y_train, y_val = train_test_split(X_train_tfidf, y_binary, test_size=0.1, random_state=42, stratify=y_binary)
    
    # Define XGBClassifier without class weights
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    
    # Hyperparameter tuning using GridSearchCV
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5, 7]
    }
    grid_search = GridSearchCV(model, param_grid, scoring='f1', cv=3, verbose=1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    models[label] = best_model
    
    # Evaluate model on the validation set (optional, can be removed)
    y_pred = best_model.predict(X_val)
    print(f"Class {label} - Accuracy: {accuracy_score(y_val, y_pred)}, "
          f"Precision: {precision_score(y_val, y_pred, average='macro')}, "
          f"Recall: {recall_score(y_val, y_pred, average='macro')}, "
          f"F1 Score: {f1_score(y_val, y_pred, average='macro')}")

# Meta-learner: Train a Gradient Boosting model using the predictions of XGB models
def train_meta_learner(models, X, y):
    meta_features = np.column_stack([model.predict_proba(X)[:, 1] for model in models.values()])
    meta_learner = GradientBoostingClassifier()  # Change this line to use a different ensemble model if needed
    
    # Normalize class labels for the meta-learner
    y_normalized = y - 1
    
    meta_learner.fit(meta_features, y_normalized)
    return meta_learner

# Prepare meta-features for the meta-learner
meta_features_train = np.column_stack([model.predict_proba(X_train_tfidf)[:, 1] for model in models.values()])
meta_features_test = np.column_stack([model.predict_proba(X_test_tfidf)[:, 1] for model in models.values()])

# Train the meta-learner
meta_learner = train_meta_learner(models, X_train_tfidf, train_df['severity'])

# Predict with meta-learner
predictions_normalized = meta_learner.predict(meta_features_test)
final_predictions = predictions_normalized + 1

# Evaluate the ensemble model
ensemble_accuracy = accuracy_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1)
ensemble_precision = precision_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_recall = recall_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_f1 = f1_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')

print(f"Ensemble Model - Accuracy: {ensemble_accuracy}, "
      f"Precision: {ensemble_precision}, "
      f"Recall: {ensemble_recall}, "
      f"F1 Score: {ensemble_f1}")

# Map numeric labels back to string labels
reverse_severity_mapping = {v: k for k, v in severity_mapping.items()}
predicted_severities = [reverse_severity_mapping[label] for label in final_predictions]

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'bug_id': test_df['bug_id'],
    'severity': predicted_severities  # Change column name to 'severity'
})

# Save to CSV
submission_path = "/Users/eren/Desktop/412Project/submission81.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")

[nltk_data] Downloading package wordnet to /Users/eren/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 1 - Accuracy: 0.9725625, Precision: 0.8078541938481887, Recall: 0.5099973141803875, F1 Score: 0.5127348995511737
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 2 - Accuracy: 0.992625, Precision: 0.8297069734483764, Recall: 0.5166036943744753, F1 Score: 0.5298949932820799
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 3 - Accuracy: 0.9808125, Precision: 0.8904345107846202, Recall: 0.5064197454717407, F1 Score: 0.5078538490247501
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 4 - Accuracy: 0.86275, Precision: 0.8452004094810615, Recall: 0.7142441025079614, F1 Score: 0.7518566680723602
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 5 - Accuracy: 0.9640625, Precision: 0.9196428571428572, Recall: 0.5287632294482216, F1 Score: 0.5450943743749683
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Class 6 - Accuracy: 0.9955, Precision: 0.6